# UNGA Voting Indicator Audit

This notebook validates the processed UN General Assembly voting
indicator generated by `src/data/process_unga.py`.

The baseline measure uses resolutions on which the United States and
China cast different substantive votes. The strict Yes-versus-No
measure is retained as a robustness specification.

The notebook checks key uniqueness, score bounds, participation
arithmetic, annual coverage, 2026 resolution inclusion, source
exceptions, and agreement between the broad and strict specifications.

In [1]:
# Complete runnable example: UNGA processed-data diagnostic

from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 1. Locate the repository and processed files
# ------------------------------------------------------------

current_directory = Path.cwd()

if (current_directory / "data" / "processed").exists():
    repository_root = current_directory
elif (current_directory.parents[1] / "data" / "processed").exists():
    repository_root = current_directory.parents[1]
else:
    raise FileNotFoundError(
        "Could not locate data/processed from the current notebook location."
    )

processed_directory = repository_root / "data" / "processed"

scores_path = (
    processed_directory
    / "unga_country_year_scores.csv"
)

audit_path = (
    processed_directory
    / "unga_anchor_resolution_audit.csv"
)

coverage_path = (
    processed_directory
    / "unga_coverage_by_year.csv"
)

exceptions_path = (
    processed_directory
    / "unga_ingestion_exceptions.csv"
)

duplicates_path = (
    processed_directory
    / "unga_duplicate_audit.csv"
)

long_votes_path = (
    processed_directory
    / "unga_votes_long.parquet"
)


# ------------------------------------------------------------
# 2. Load the processed outputs
# ------------------------------------------------------------

scores = pd.read_csv(scores_path)

audit = pd.read_csv(audit_path)

coverage = pd.read_csv(coverage_path)

exceptions = pd.read_csv(exceptions_path)

duplicates = pd.read_csv(duplicates_path)

long_votes = pd.read_parquet(long_votes_path)


# ------------------------------------------------------------
# 3. Structural checks
# ------------------------------------------------------------

duplicate_resolution_keys = int(
    audit.duplicated(
        [
            "year",
            "resolution_id",
        ]
    ).sum()
)

duplicate_country_year_keys = int(
    scores.duplicated(
        [
            "country_key",
            "year",
        ]
    ).sum()
)

broad_scores = (
    scores["unga_score_broad"]
    .dropna()
)

strict_scores = (
    scores["unga_score_strict_yn"]
    .dropna()
)

count_identity_failures = int(
    (
        scores["votes_with_us"]
        + scores["votes_with_china"]
        + scores[
            "neutral_third_position_votes"
        ]
        != scores[
            "broad_participated_votes"
        ]
    ).sum()
)

participation_order_failures = int(
    (
        scores["broad_participated_votes"]
        > scores[
            "broad_eligible_resolutions"
        ]
    ).sum()
)

participation_rate_failures = int(
    (
        scores["broad_participation_rate"]
        .dropna()
        .between(
            0,
            1,
            inclusive="both",
        )
        == False
    ).sum()
)

coverage_years_match = (
    set(coverage["year"])
    == set(scores["year"].unique())
)


# ------------------------------------------------------------
# 4. Check 2026 resolution inclusion
# ------------------------------------------------------------

expected_2026_resolutions = {
    "A/RES/80/247",
    "A/RES/80/248",
    "A/RES/80/249",
    "A/RES/80/250",
    "A/RES/80/251",
    "A/RES/80/254",
    "A/RES/80/256",
    "A/RES/80/257",
    "A/RES/80/258",
    "A/RES/80/263",
    "A/RES/80/264",
    "A/RES/80/265",
    "A/RES/80/267",
    "A/RES/80/268",
    "A/RES/80/269",
    "A/RES/80/282",
    "A/RES/80/286",
    "A/RES/80/287",
    "A/RES/80/289",
    "A/RES/ES-11/10",
}

processed_2026_resolutions = set(
    audit.loc[
        audit["year"].eq(2026),
        "resolution_id",
    ]
)

missing_2026_resolutions = sorted(
    expected_2026_resolutions
    - processed_2026_resolutions
)

unexpected_2026_resolutions = sorted(
    processed_2026_resolutions
    - expected_2026_resolutions
)


# ------------------------------------------------------------
# 5. Specification comparison
# ------------------------------------------------------------

comparison_data = scores[
    [
        "unga_score_broad",
        "unga_score_strict_yn",
    ]
].dropna()

pearson_correlation = (
    comparison_data[
        "unga_score_broad"
    ]
    .corr(
        comparison_data[
            "unga_score_strict_yn"
        ],
        method="pearson",
    )
)

spearman_correlation = (
    comparison_data[
        "unga_score_broad"
    ]
    .corr(
        comparison_data[
            "unga_score_strict_yn"
        ],
        method="spearman",
    )
)

scores["broad_strict_difference"] = (
    scores["unga_score_broad"]
    - scores["unga_score_strict_yn"]
).abs()

largest_specification_differences = (
    scores[
        [
            "country_name",
            "year",
            "unga_score_broad",
            "unga_score_strict_yn",
            "broad_strict_difference",
            "broad_participated_votes",
            "strict_participated_votes",
        ]
    ]
    .dropna(
        subset=[
            "unga_score_broad",
            "unga_score_strict_yn",
        ]
    )
    .sort_values(
        "broad_strict_difference",
        ascending=False,
    )
    .head(15)
)


# ------------------------------------------------------------
# 6. Critical assertions
# ------------------------------------------------------------

assert duplicate_resolution_keys == 0

assert duplicate_country_year_keys == 0

assert broad_scores.between(
    -1,
    1,
    inclusive="both",
).all()

assert strict_scores.between(
    -1,
    1,
    inclusive="both",
).all()

assert count_identity_failures == 0

assert participation_order_failures == 0

assert participation_rate_failures == 0

assert coverage_years_match

assert not missing_2026_resolutions

assert not unexpected_2026_resolutions


# ------------------------------------------------------------
# 7. Print one consolidated report
# ------------------------------------------------------------

print("=" * 70)
print("UNGA INDICATOR DIAGNOSTIC REPORT")
print("=" * 70)

print("\nPANEL")
print(
    "Years:",
    f"{scores['year'].min()}-"
    f"{scores['year'].max()}",
)
print(
    "Country-years:",
    f"{len(scores):,}",
)
print(
    "Country-resolution observations:",
    f"{len(long_votes):,}",
)
print(
    "Unique resolution-years:",
    f"{len(audit):,}",
)

print("\nSCORE AVAILABILITY")
print(
    "Available baseline scores:",
    int(
        (
            scores["score_status"]
            == "available"
        ).sum()
    ),
)
print(
    "Provisionally insufficient:",
    int(
        (
            scores["score_status"]
            == (
                "provisionally_"
                "insufficient_participation"
            )
        ).sum()
    ),
)
print(
    "Missing broad scores:",
    int(
        scores["unga_score_broad"]
        .isna()
        .sum()
    ),
)

print("\nSCORE RANGES")
print(
    "Broad minimum:",
    broad_scores.min(),
)
print(
    "Broad maximum:",
    broad_scores.max(),
)
print(
    "Strict minimum:",
    strict_scores.min(),
)
print(
    "Strict maximum:",
    strict_scores.max(),
)

print("\nSTRUCTURAL VALIDATION")
print(
    "Duplicate resolution keys:",
    duplicate_resolution_keys,
)
print(
    "Duplicate country-year keys:",
    duplicate_country_year_keys,
)
print(
    "Count identity failures:",
    count_identity_failures,
)
print(
    "Participation-order failures:",
    participation_order_failures,
)
print(
    "Participation-rate failures:",
    participation_rate_failures,
)
print(
    "Coverage years match:",
    coverage_years_match,
)

print("\nSOURCE QUALITY")
print(
    "Recorded ingestion exceptions:",
    len(exceptions),
)
print(
    "Duplicate audit rows:",
    len(duplicates),
)
print(
    "Duplicate audit conflicts:",
    int(
        duplicates[
            "conflict_at_top_priority"
        ].fillna(False).sum()
    ),
)

print("\n2026 RESOLUTION CHECK")
print(
    "Expected resolutions:",
    len(expected_2026_resolutions),
)
print(
    "Processed resolutions:",
    len(processed_2026_resolutions),
)
print(
    "Missing resolutions:",
    missing_2026_resolutions,
)
print(
    "Unexpected resolutions:",
    unexpected_2026_resolutions,
)

print("\nBROAD VERSUS STRICT SPECIFICATIONS")
print(
    "Pearson correlation:",
    f"{pearson_correlation:.4f}",
)
print(
    "Spearman correlation:",
    f"{spearman_correlation:.4f}",
)

print("\nLARGEST BROAD-STRICT DIFFERENCES")
print(
    largest_specification_differences
    .to_string(index=False)
)

print("\nLATEST-YEAR COVERAGE")
print(
    coverage.loc[
        coverage["year"] >= 2023
    ].to_string(index=False)
)

print("\nINGESTION EXCEPTIONS")
print(
    exceptions.to_string(index=False)
)

print("\nAll critical UNGA assertions passed.")

UNGA INDICATOR DIAGNOSTIC REPORT

PANEL
Years: 2004-2026
Country-years: 4,392
Country-resolution observations: 377,895
Unique resolution-years: 1,959

SCORE AVAILABILITY
Available baseline scores: 4201
Provisionally insufficient: 174
Missing broad scores: 17

SCORE RANGES
Broad minimum: -1.0
Broad maximum: 1.0
Strict minimum: -1.0
Strict maximum: 1.0

STRUCTURAL VALIDATION
Duplicate resolution keys: 0
Duplicate country-year keys: 0
Count identity failures: 0
Participation-order failures: 0
Participation-rate failures: 0
Coverage years match: True

SOURCE QUALITY
Recorded ingestion exceptions: 3
Duplicate audit rows: 193
Duplicate audit conflicts: 0

2026 RESOLUTION CHECK
Expected resolutions: 20
Processed resolutions: 20
Missing resolutions: []
Unexpected resolutions: []

BROAD VERSUS STRICT SPECIFICATIONS
Pearson correlation: 0.9806
Spearman correlation: 0.9545

LARGEST BROAD-STRICT DIFFERENCES
            country_name  year  unga_score_broad  unga_score_strict_yn  broad_strict_differ